# Task 3: GPU category articles with Qwen

Select a notebook kernel with the project dependencies from `requirements.txt` installed and CUDA-enabled PyTorch available. This uses Qwen3-4B-Instruct-2507 in 4-bit NF4 with BF16 computation on the local NVIDIA GPU. The first load downloads the original model weights (about 8 GB) and quantizes them during loading. Reviews and inference remain local.

The five Task 2 categories are retained. Metadata names are inconsistent, so article instructions require cautious profile comparisons and prevent guessing a product identity. Rankings use source-row statistics; sampled evidence excerpts are at most 160 model tokens each. Full texts and source positions are saved for checking. This is inference, not fine-tuning.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'summarize_categories_qwen.py').exists() else Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import torch
assert torch.cuda.is_available(), 'Select the .venv-qwen kernel with CUDA PyTorch.'
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)
from summarize_categories_qwen import QwenWriter, run, prepare_data, VARIANTS
from IPython.display import Markdown, display
df = prepare_data(ROOT / 'data/reviews_with_meta_categories.csv')
writer = QwenWriter()


## Generate five articles

The cited prompt requires source IDs, reviewer attribution, and explicit uncertainty about product identity. Evidence sampling spans positive, negative, and neutral ratings plus leading and lowest-rated eligible profiles. Excerpts do not establish prevalence across all reviews. Citation ID validity is checked automatically; whether a citation supports a claim still requires manual evaluation.


In [ ]:
OUTPUT = ROOT / 'outputs/qwen_category_articles'
manifest = run(df, writer, OUTPUT)
assert len(manifest) == df['meta_category'].nunique()
for item in manifest:
    display(Markdown((OUTPUT / item['article']).read_text(encoding='utf-8')))


## Compare three prompts on identical evidence

The prompts vary article-writing instructions while holding model, evidence and greedy decoding fixed. Compare basic, evidence-grounded, and evidence-grounded with citations. Record factuality, complaint coverage, readability, product attribution, invented specifications and citation support. Keep BART's articles as a baseline; its samples differ, so this is a qualitative comparison rather than a controlled model benchmark.


In [ ]:
comparison = run(df, writer, ROOT / 'experiments/qwen_prompt_comparison',
                 variants=tuple(VARIANTS), category='Cases and protective covers')
for item in comparison:
    display(Markdown((ROOT / 'experiments/qwen_prompt_comparison' / item['article']).read_text(encoding='utf-8')))
    

## Evaluation

Read each article together with its JSON. Confirm source-row attribution, statistics and citations. Do not treat a missing or conflicting metadata name as a verified product identity. The lowest mean rating is not evidence that a product is unsafe or should be avoided. Fine-tune only after constructing human-reviewed targets and splitting by verified product identity to prevent leakage.


## Completed local GPU run

Five articles and three cover-category prompt variants were generated on the RTX 5060 Laptop GPU in 4-bit precision. Article generation took about 43?93 seconds each, with PyTorch peak allocations of 5.1?7.5 GiB. The largest prompt used 3,867 input tokens. CUDA computation and 4-bit kernels passed their smoke checks.

Open `../outputs/qwen_category_articles/index.md` for the articles, computed statistics and source evidence. `quality_review.md` records runtime measurements and factual errors observed during inspection. All evidence IDs were valid, but Qwen still produced rating and profile-attribution mistakes and unsupported quotations. The cited prompt is retained for traceability; factual quality still requires manual review.
